# 02 · 조달청 시설공통자재 가격정보 — 기계설비

| | |
|---|---|
| **목적** | 신품 단가 기준선 확보. `supplier_parts` 단가 그라운딩에 쓸 수 있는지 **검증** |
| **input** | `data/raw/시설공통자재(기계설비).xlsx` (읽기 전용) |
| **output** | `data-analysis/data/processed/kseis_machinery.parquet`<br>`data-analysis/reports/kseis_machinery_decisions.json` |

> ⚠️ **헤더가 7행째에 있다.** 1~6행은 파일 제목·생성일 머리글이라 `skiprows=6` 없이 읽으면
> 컬럼명이 `Unnamed: 0` 이 된다.

In [ ]:
import sys
sys.path.insert(0, "..")
import pandas as pd
from common import RAW, profile, save_decisions, save_processed, top_values

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)


## 1. 개요

| 항목 | 값 |
|---|---|
| 출처 | [KSEIS 2026 상반기 조달청 가격정보](https://www.kseis.co.kr/bbs/data/dataDetail.do?bbs_seq=67467289780142) |
| 원배포 | 조달청 — `조달청_시설공통자재 가격정보 내역` |
| 수집일 | 2026-08-07 (파일 생성일 2026-04-21 · 게시일자 2026-04-20) |
| 분야 | **기계설비** |
| 왜 받았나 | 전동기·펌프 등 신품 단가 기준선 |

In [ ]:
XLSX = RAW / "시설공통자재(기계설비).xlsx"
print("파일:", XLSX.name)
print("크기:", f"{XLSX.stat().st_size:,} bytes")

머리글 = pd.read_excel(XLSX, header=None, nrows=7)
print("\n1~7행 (머리글) —")
print(머리글.iloc[:, 0].to_string())
print("\n→ 7행째(index 6)가 실제 컬럼명이다. skiprows=6.")

## 2. 로드 + 기초 통계

In [ ]:
df_raw = pd.read_excel(XLSX, skiprows=6)
print(f"원본 shape: {df_raw.shape[0]:,}행 × {df_raw.shape[1]}열")

In [ ]:
profile(df_raw)

In [ ]:
df_raw.head(5)

## 3. EDA

### 3-1. 🚨 `자재구분` 에 **끝 공백**이 있다

`'시설자재(기계설비분야) '` — 마지막에 공백 1칸. 중진공 CSV 의 `'환경  설비'`(공백 2칸)와
**같은 종류의 함정**이다. `==` 비교나 join 키로 쓰면 조용히 어긋난다.

In [ ]:
print("자재구분 고유값 (repr) —")
for v in df_raw["자재구분"].unique():
    print(f"  {v!r}   길이 {len(v)} · strip 후 길이 {len(v.strip())}")
print()
print("⚠ 끝 공백을 그대로 두면 '시설자재(기계설비분야)' 와 매칭되지 않는다.")

### 3-2. 품명·단위 분포

In [ ]:
print(f"품명 고유값: {df_raw['품명'].nunique()}종 / {len(df_raw):,}행")
print()
print(top_values(df_raw, "품명", 10).to_string(index=False))
print()
print(top_values(df_raw, "단위", 6).to_string(index=False))

### 3-3. 가격 분포·이상치

In [ ]:
가격 = df_raw["가격"]
print(가격.describe(percentiles=[0.1, 0.5, 0.9, 0.99]).apply(lambda v: f"{v:,.0f}").to_string())
print()
print(f"0원 이하: {(가격 <= 0).sum()}건 · 결측: {가격.isna().sum()}건")
print("\n최고가 5건 —")
print(df_raw.nlargest(5, "가격")[["품명", "규격", "단위", "가격"]].to_string(index=False))

### 3-4. 🚨 MaintQ 부품과 실제로 겹치는가 — **검증**

`data_list.md` 는 이 파일을 *"전동기·펌프 등 신품 단가 기준선"* 으로 적어 놨다. **정말 그런지 품명으로 직접 확인한다.**
**특히 `펌프` 가 정말 있는지 확인한다** — 문서는 있다고 적어 놨다.

In [ ]:
키워드 = ["냉각팬", "팬", "PCB", "제어보드", "전원모듈", "모듈", "모터", "전동기",
        "펌프", "케이블", "냉각", "인버터", "베어링", "필터", "히트싱크"]
print(f"{'키워드':<10}{'건수':>6}  실제 품명 예시")
print("-" * 78)
for kw in 키워드:
    hit = df_raw[df_raw["품명"].str.contains(kw, na=False)]
    예 = sorted({p.replace(chr(10), " ") for p in hit["품명"]})[:2]
    print(f"{kw:<10}{len(hit):>6}  {예 if 예 else '— 없음'}")

> 🚨 **문자열 포함 검색은 오탐을 낸다.** `'팬'` 이 걸리면 `익스팬션죠인트` 같은
> **전혀 다른 품목**이 잡힌다. 건수만 보고 "팬이 있다"고 판단하면 안 되고 품명을 봐야 한다.

## 4. 전처리

In [ ]:
df = df_raw.copy()

# 1) 🚨 문자열 끝/앞 공백 제거 — 단, 원본 값을 별도 컬럼에 보존한다
df["자재구분_원본"] = df["자재구분"]
for c in ["자재구분", "품명", "규격", "단위", "부가세여부", "인도조건", "관할지역"]:
    df[c] = df[c].astype(str).str.strip()

# 2) 품명 안의 줄바꿈 제거 ('익스팬션죠인트\n스테인레스(단식)')
df["품명"] = df["품명"].str.replace(r"\s*\n\s*", " ", regex=True)
df["규격"] = df["규격"].str.replace(r"\s*\n\s*", " ", regex=True)

# 3) 타입
df["가격"] = pd.to_numeric(df["가격"], errors="coerce")
df["게시일자"] = pd.to_datetime(df["게시일자"], errors="coerce")
df["물품분류번호"] = df["물품분류번호"].astype(str)
df["물품식별번호"] = df["물품식별번호"].astype(str)

# 4) 파생 — 분야 태그 (90 에서 두 파일을 세로로 합칠 때 출처 구분용)
df["분야"] = "기계설비" if "기계설비" in df["자재구분"].iloc[0] else "전기정보통신"

df = df.drop(columns=["NO"])
print(f"정제 후: {df.shape[0]:,}행 × {df.shape[1]}열")
print("분야 태그:", df["분야"].unique())

## 5. 결정 로그

In [ ]:
column_decisions = [
    {"column": "품명", "used": True, "reason": "부품 매칭의 1차 키. 줄바꿈 제거 + strip 필요"},
    {"column": "규격", "used": True, "reason": "같은 품명 안에서 단가를 가르는 유일한 축"},
    {"column": "단위", "used": True, "reason": "가격 비교의 전제. m/개/대 가 섞여 있어 단위 무시 비교는 무의미"},
    {"column": "가격", "used": True, "reason": "신품 단가 기준선 본체. 부가세 별도"},
    {"column": "자재구분", "used": True,
     "reason": "분야 구분. ⛔ 끝 공백 1칸이 있어 strip 필수 — 원본은 자재구분_원본 에 보존"},
    {"column": "물품분류번호", "used": True, "reason": "조달청 품목 분류 코드. 카테고리 매핑의 후보 키"},
    {"column": "물품식별번호", "used": True, "reason": "행 고유 식별자 — 중복 검사에 사용"},
    {"column": "게시일자", "used": True, "reason": "단가 스냅샷 시점 (D31 단가 스냅샷 개념과 같은 성격)"},
    {"column": "NO", "used": False, "reason": "엑셀 행번호. 파일 내 순번일 뿐 의미 없음 — 제거"},
    {"column": "부가세여부", "used": False, "reason": "전건 '부가가치세별도' — 상수라 변별력 0"},
    {"column": "인도조건", "used": False, "reason": "전건 '납품장소도' — 상수"},
    {"column": "관할지역", "used": False, "reason": "'전지역(제주제외)' 편중 · MaintQ 판정에 지역 축이 없다"},
]
assert len(column_decisions) == df_raw.shape[1], \
    f"결정 {len(column_decisions)} != 원본 컬럼 {df_raw.shape[1]}"
print(f"컬럼 결정 {len(column_decisions)}건 — 원본 컬럼 수와 일치")

In [ ]:
겹침 = {}
for kw in ["냉각팬", "PCB", "제어보드", "전원모듈", "모터", "전동기", "펌프", "케이블",
           "인버터", "베어링", "필터", "히트싱크"]:
    겹침[kw] = int(df["품명"].str.contains(kw, na=False).sum())

save_decisions(
    "kseis_machinery",
    column_decisions,
    source_url="https://www.kseis.co.kr/bbs/data/dataDetail.do?bbs_seq=67467289780142",
    collected_at="2026-08-07",
    file_generated="2026-04-21",
    분야="기계설비",
    row_count_raw=int(df_raw.shape[0]),
    col_count_raw=int(df_raw.shape[1]),
    row_count_processed=int(df.shape[0]),
    generated_columns=["자재구분_원본", "분야"],
    maintq_part_keyword_hits=겹침,
    caveats=[
        "헤더가 7행째 — skiprows=6 없이 읽으면 컬럼이 전부 Unnamed",
        "자재구분에 끝 공백 1칸이 있다 ('시설자재(...분야) ') — join 키로 쓰면 조용히 미스",
        "품명·규격에 줄바꿈(\\n)이 들어 있다",
        "공공조달 단가라 규격·수량이 민간과 달라 그대로 시장 신품가가 아니다",
        "가격은 부가세 별도",
    ],
)

## 6. 정제 데이터 저장

In [ ]:
save_processed("kseis_machinery", df)